# SmartSpend AI: Personal Expense Intelligence

This notebook performs end-to-end analysis of personal expense data using the
`SmartSpend_AI_Expense_Dataset.csv` dataset (1 208 transactions, Sep 2025 – Aug 2026).

**Sections covered:**
1. Data loading
2. Data cleaning
3. Exploratory Data Analysis (EDA)
4. Key Performance Indicators (KPIs)
5. Anomaly Detection – Isolation Forest
6. Monthly Spending Forecast – Linear Regression
7. Key Findings
8. Conclusion

## 1. Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

%matplotlib inline
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

df_raw = pd.read_csv('SmartSpend_AI_Expense_Dataset.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head()

ModuleNotFoundError: No module named 'pandas'

## 2. Data Cleaning

In [ ]:
df = df_raw.copy()

# Convert Date to datetime
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Fill missing Merchant / Description
df['Merchant'] = df['Merchant'].fillna('Unknown')
df['Description'] = df['Description'].fillna('Unknown')

# Remove duplicate rows
before = len(df)
df = df.drop_duplicates()
print(f'Removed {before - len(df)} duplicate rows')

# Ensure Amount_INR is numeric; drop rows where it cannot be parsed
df['Amount_INR'] = pd.to_numeric(df['Amount_INR'], errors='coerce')
df = df.dropna(subset=['Amount_INR', 'Date'])

# Derive YearMonth for grouping
df['YearMonth'] = df['Date'].dt.to_period('M').astype(str)

print(f'Clean dataset shape: {df.shape}')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
df.info()

In [ ]:
# Check missing values in cleaned dataset
print('Missing values per column:')
df.isnull().sum()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics for Amount_INR
df['Amount_INR'].describe().round(2)

In [ ]:
# Monthly spending
monthly = df.groupby('YearMonth')['Amount_INR'].sum().sort_index()

fig, ax = plt.subplots()
ax.bar(monthly.index, monthly.values, color='#3b82d4')
ax.set_xlabel('Month')
ax.set_ylabel('Total Spending (₹)')
ax.set_title('Monthly Spending')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Category-wise spending
cat_spending = df.groupby('Category')['Amount_INR'].sum().sort_values(ascending=False)

fig, ax = plt.subplots()
ax.barh(cat_spending.index, cat_spending.values, color='#7c5cd8')
ax.set_xlabel('Total Spending (₹)')
ax.set_title('Spending by Category')
plt.tight_layout()
plt.show()

In [ ]:
# Transaction amount distribution
fig, ax = plt.subplots()
ax.hist(df['Amount_INR'], bins=40, color='#3b82d4', edgecolor='white')
ax.set_xlabel('Amount (₹)')
ax.set_ylabel('Number of Transactions')
ax.set_title('Distribution of Transaction Amounts')
plt.tight_layout()
plt.show()

In [ ]:
# Payment method breakdown
payment_counts = df['Payment_Method'].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(payment_counts.index, payment_counts.values, color='#3b82d4')
ax.set_xlabel('Payment Method')
ax.set_ylabel('Number of Transactions')
ax.set_title('Transactions by Payment Method')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 4. Key Performance Indicators (KPIs)

In [ ]:
total_spending    = df['Amount_INR'].sum()
avg_transaction   = df['Amount_INR'].mean()
median_transaction= df['Amount_INR'].median()
num_transactions  = len(df)
highest_category  = df.groupby('Category')['Amount_INR'].sum().idxmax()
highest_cat_amt   = df.groupby('Category')['Amount_INR'].sum().max()
monthly_totals    = df.groupby('YearMonth')['Amount_INR'].sum()
highest_month     = monthly_totals.idxmax()
highest_month_amt = monthly_totals.max()

print('=' * 45)
print('         KEY PERFORMANCE INDICATORS')
print('=' * 45)
print(f'  Total Spending          : ₹{total_spending:>12,.2f}')
print(f'  Total Transactions      : {num_transactions:>12,}')
print(f'  Average Transaction     : ₹{avg_transaction:>12,.2f}')
print(f'  Median Transaction      : ₹{median_transaction:>12,.2f}')
print(f'  Highest-Spend Category  : {highest_category} (₹{highest_cat_amt:,.2f})')
print(f'  Highest-Spend Month     : {highest_month} (₹{highest_month_amt:,.2f})')
print('=' * 45)

## 5. Anomaly Detection – Isolation Forest

Isolation Forest is an unsupervised algorithm that isolates observations by randomly
selecting a feature and a split value. Transactions that require fewer splits to isolate
score as anomalies. Here we flag them as **unusual spending** — not fraud.

In [ ]:
X = df[['Amount_INR']].values
iso = IsolationForest(contamination=0.05, random_state=42)
df['anomaly_flag'] = iso.fit_predict(X)   # -1 unusual, 1 normal

unusual = df[df['anomaly_flag'] == -1]
normal  = df[df['anomaly_flag'] ==  1]

print(f'Total transactions  : {len(df):,}')
print(f'Normal transactions : {len(normal):,}')
print(f'Unusual transactions: {len(unusual):,}')

In [ ]:
# Scatter plot – normal vs unusual
fig, ax = plt.subplots()
ax.scatter(normal['Date'],  normal['Amount_INR'],  s=10, color='#3b82d4', alpha=0.5, label='Normal')
ax.scatter(unusual['Date'], unusual['Amount_INR'], s=30, color='#e03e3e', alpha=0.8, label='Unusual')
ax.set_xlabel('Date')
ax.set_ylabel('Amount (₹)')
ax.set_title('Transaction Amounts – Normal vs Unusual')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 unusual transactions
top10_unusual = (
    unusual.sort_values('Amount_INR', ascending=False)
    .head(10)[['Date', 'Category', 'Merchant', 'Description', 'Amount_INR', 'Payment_Method']]
    .reset_index(drop=True)
)
print('Top 10 Unusual / High-Value Transactions')
top10_unusual

## 6. Monthly Spending Forecast – Linear Regression

> **Note:** This is an illustrative trend-based estimate. The dataset covers
> approximately one year of data (Sep 2025 – Aug 2026).
> The **last 3 months are held out as a test set**; MAE and RMSE are calculated
> on those unseen months only. The final model is then **retrained on all 12 months**
> to produce the next-month estimate. The estimate should be treated as a
> directional indicator, not a precise prediction.

In [ ]:
# Aggregate monthly spending and create sequential month number
monthly_df = (
    df.groupby('YearMonth')['Amount_INR']
    .sum()
    .sort_index()
    .reset_index()
)
monthly_df.columns = ['YearMonth', 'Total_Spending']
monthly_df['Month_Num'] = np.arange(1, len(monthly_df) + 1)

# Train / test split: first 9 months train, last 3 months test
train_df = monthly_df.iloc[:9]
test_df  = monthly_df.iloc[9:]

X_tr = train_df['Month_Num'].values.reshape(-1, 1)
y_tr = train_df['Total_Spending'].values
X_te = test_df['Month_Num'].values.reshape(-1, 1)
y_te = test_df['Total_Spending'].values

# Fit on training set, evaluate on unseen test set
eval_model = LinearRegression()
eval_model.fit(X_tr, y_tr)
y_te_pred = eval_model.predict(X_te)

mae  = mean_absolute_error(y_te, y_te_pred)
rmse = np.sqrt(mean_squared_error(y_te, y_te_pred))

# Retrain final model on all 12 months, then forecast next month
X_all = monthly_df['Month_Num'].values.reshape(-1, 1)
y_all = monthly_df['Total_Spending'].values
final_model = LinearRegression()
final_model.fit(X_all, y_all)
y_pred_all = final_model.predict(X_all)   # trend line over full history

next_month_num      = len(monthly_df) + 1
next_month_forecast = final_model.predict(np.array([[next_month_num]]))[0]
last_period         = pd.Period(monthly_df['YearMonth'].iloc[-1], freq='M')
next_period         = last_period + 1

print(f'MAE  (test – last 3 months): ₹{mae:,.2f}')
print(f'RMSE (test – last 3 months): ₹{rmse:,.2f}')
print(f'Forecast for {next_period} (final model, all months): ₹{next_month_forecast:,.2f}')

In [ ]:
# Plot historical spending + trend + forecast
labels = list(monthly_df['YearMonth']) + [str(next_period)]
trend_extended = list(y_pred_all) + [next_month_forecast]

x_pos = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(12, 5))
# Training months
ax.bar(x_pos[:9],  train_df['Total_Spending'], color='#3b82d4', alpha=0.85, label='Train (months 1–9)')
# Test months
ax.bar(x_pos[9:-1], test_df['Total_Spending'], color='#7c5cd8', alpha=0.85, label='Test (months 10–12)')
# Forecast bar
ax.bar(x_pos[-1],  next_month_forecast, color='#f59e0b', alpha=0.85, label=f'Forecast ({next_period})')
# Trend line from final model
ax.plot(x_pos, trend_extended, color='#e03e3e', linewidth=2, marker='o', markersize=5,
        label='Trend (final model, all months)')
ax.set_xticks(x_pos)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_xlabel('Month')
ax.set_ylabel('Total Spending (₹)')
ax.set_title('Monthly Spending — Train/Test Split, Trend & Forecast')
ax.legend()
plt.tight_layout()
plt.show()

monthly_df[['YearMonth','Total_Spending']].assign(
    Total_Spending=lambda x: x['Total_Spending'].round(2)
)

## 7. Key Findings

In [ ]:
# Category breakdown summary
cat_summary = (
    df.groupby('Category')['Amount_INR']
    .agg(total='sum', count='count', mean='mean')
    .sort_values('total', ascending=False)
    .round(2)
)
cat_summary['share_%'] = (cat_summary['total'] / cat_summary['total'].sum() * 100).round(1)
print('Category Breakdown:')
cat_summary

In [ ]:
# Weekend vs weekday spending
weekend_summary = df.groupby('Weekend')['Amount_INR'].agg(total='sum', count='count', mean='mean').round(2)
print('Weekend vs Weekday Spending:')
weekend_summary

## 8. Conclusion

This notebook demonstrated a complete personal expense analytics pipeline:

| Step | Technique | Outcome |
|------|-----------|---------|
| Data cleaning | pandas datetime / fillna / drop_duplicates | Clean dataset ready for analysis |
| EDA | Matplotlib bar/histogram charts | Spending patterns by month and category |
| KPIs | pandas groupby aggregation | Total spend, avg transaction, top category/month |
| Anomaly detection | Isolation Forest (scikit-learn) | ~5% flagged as unusual transactions |
| Forecast | Linear Regression (scikit-learn) | Trend-based next-month estimate with MAE & RMSE |

**Limitations:**  
- Only ~12 months of data is available, which limits forecast reliability.  
- Linear Regression captures only the overall trend; it ignores seasonality.  
- Anomaly detection flags statistical outliers, not confirmed fraud.

The Streamlit application (`app.py`) provides an interactive interface for the same analytics.